# 📗 SFT 지도 파인튜닝: SFTTrainer로 노무 상담 모델 학습하기

SFT(Supervised Fine-Tuning, 지도 파인튜닝)는 질문과 정답 답변 쌍으로 모델을 더 학습시켜 그 답처럼 답하게 만드는 방법입니다. 이 교안은 지난 시간에 만든 노무 상담 데이터로 `Qwen/Qwen3.5-0.8B`를 학습하고, 학습률과 에폭을 바꿔 가며 같은 질문의 답을 비교합니다. 에폭마다 저장한 체크포인트는 교안 02가 채점해 쓸 모델을 고릅니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: SFTTrainer로 노무 상담 모델 학습하기'이고 아래 띠는 'Qwen3.5-0.8B · 지난 시간에 만든 노무 상담 데이터'입니다. 네 판이 1에서 4로 이어집니다. 1 '학습 전 모델의 답'(학습 2 · 검증 1 · 범위 밖 1 · 일반 2)은 '학습', '검증', '범위 밖', '일반' 질문 카드 묶음이 학습 전 모델의 답 말풍선으로 가고 그 옆 '정답 답변'과 비교하는 그림, 2 '학습 전 점검'(labels -100 · 대화 끝 토큰 · max_length)은 토큰 칸 한 줄로, 회색 '질문' 칸 위에 '-100: 손실 제외', 초록 '답변' 칸과 맨 끝 '대화 끝 토큰' 칸 위에 '손실 계산' 화살표가 있고 칸 줄 전체 아래에 'max_length 이내' 화살표가 있습니다. 3 '첫 학습과 학습 후 답'(lr 2e-5 · 1에폭 · 같은 6개 질문)은 갱신 횟수에 따라 내려가는 학습 손실 그래프와 Base 말풍선에서 Tuned 말풍선으로 가는 화살표, 4 '학습률과 에폭 늘리기'(lr 2e-5 · lr 5e-6 · 4에폭)는 'lr 2e-5'와 'lr 5e-6' 카드와, '에폭마다 저장' 시간 줄 위에 놓인 '1에폭'부터 '4에폭'까지의 폴더 네 개입니다.">

**학습 목표** (그림의 번호와 같습니다)

1. **학습 전 모델의 답을 정답 답변과 비교해 SFT로 가르칠 것을 정합니다.**
2. **학습 전에 답변 토큰만 손실에 들어가는지, 대화 끝 토큰과 `max_length`가 맞는지 확인합니다.**
    - **`labels` -100:** 손실 계산에서 빼는 토큰 표시입니다. 답변 앞의 토큰 전부에 붙습니다.
    - **대화 끝 토큰 `<|im_end|>`:** 답 끝에 붙는 토큰입니다. 이 토큰까지 배워야 답을 마치고 멈춥니다.
    - **`max_length`:** 학습 예시 하나의 최대 토큰 수입니다. 넘으면 뒤쪽이 잘립니다.
3. **SFTTrainer로 학습하며 손실을 MLflow에 기록하고, 학습 전·후 답을 비교합니다.**
4. **학습률과 에폭을 바꾼 학습의 답을 비교하고, 에폭마다 체크포인트를 저장합니다.**

## 1. SFT(지도 파인튜닝)

**SFT(Supervised Fine-Tuning, 지도 파인튜닝)**
- 질문과 정답 답변 쌍을 보여 주고, 정답 답변의 토큰을 맞히도록 가중치를 더 학습하는 방법입니다. 사전학습과 같은 다음 토큰 맞히기지만, 질문은 앞 내용으로만 쓰고 답변 토큰만 맞힙니다.
- **목적:** 모델이 업무에 맞는 말투, 형식, 내용(이 교안은 노동법 조문)으로 답하게 합니다.
- **한계:** 말투, 형식, 거절 같은 행동은 잘 배우지만, 학습 데이터에 없는 질문에는 법 숫자 같은 사실을 지어내기 쉽습니다. 그래서 실무에서는 사실은 RAG로 찾아 넣고, 말투와 형식은 SFT로 맞추는 조합을 많이 씁니다.
- **Base와 Tuned:** Base는 허브의 `Qwen/Qwen3.5-0.8B`(Instruct 모델)를 그대로 쓰는 학습 전 모델, Tuned는 교안 02가 체크포인트 가운데 고를 학습 후 모델입니다. 이 교안은 학습한 모델을 학습률로 부릅니다(lr 2e-5 모델, lr 5e-6 모델). 허브의 `Qwen/Qwen3.5-0.8B-Base`는 지시 학습 전의 다른 모델이라 쓰지 않습니다.


<img src="images/01_sft_idea.png" width="1050" alt="제목은 'SFT: 질문과 정답 답변 쌍으로 답하는 법을 맞춥니다'입니다. 왼쪽 '학습 전 모델(Base)' 카드는 'Instruct 모델'이고 글머리 목록으로 길게 답하는 말풍선 아래 '일반 대화식으로 길게 답함'이 적혀 있습니다. 가운데 'SFTTrainer' 카드에는 '질문'과 '정답 답변' 쌍 세 묶음이 있습니다. 오른쪽 '학습 후 모델(Tuned)' 카드에는 체크 표시와 '조문 근거로 짧게 답함', 멈춤 표시와 '범위 밖 질문은 거절'이 있습니다. 맨 아래 띠는 '이미 답할 줄 아는 모델에게 업무 말투와 지식, 거절을 가르칩니다'입니다.">


손실은 손실 대상 토큰마다 모델이 그 토큰에 준 확률의 음의 로그를 평균한 값입니다(딥러닝 학습 원리 단원의 교차 엔트로피).

$\displaystyle L = -\frac{1}{T}\sum_{t=1}^{T} \log p(y_t \mid \text{prompt}, y_{<t})$

- **손실 대상 토큰:** 정답 답변 토큰과 그 뒤의 대화 끝 토큰 `<|im_end|>`, 줄바꿈 토큰입니다.
- **$T$, $t$:** 손실 대상 토큰의 개수와 그 순서 번호(1, 2, ..., $T$)입니다.
- **$y_t$:** $t$번째 손실 대상 토큰 **하나**입니다.
- **$y_{<t}$:** $y_t$보다 앞의 손실 대상 토큰 **전부**($t-1$개, $t=1$이면 없음)입니다.
- **prompt:** 시스템 메시지, 질문, assistant 머리(`<|im_start|>assistant`), 빈 생각 블록입니다. 앞 내용으로만 쓰고 손실에는 넣지 않습니다.
- **$p(y_t \mid \text{prompt}, y_{<t})$:** prompt와 $y_{<t}$를 모두 보고 모델이 $y_t$에 준 확률입니다. 1에 가까울수록 $L$이 작아집니다.

## 2. 실습 준비

### 2-1. 라이브러리와 경로

라이브러리와 자료 경로를 준비합니다. 계산 장치와 그래프 글꼴 코드는 지난 시간과 같습니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 2-2. 학습 전 모델

도메인 SFT는 이미 질문에 답할 줄 아는 Instruct 모델에서 시작해, 작은 업무 데이터로 말투, 형식, 업무 지식만 맞춥니다. 이 교안은 Qwen3.5 계열에서 가장 작은 Instruct 모델 `Qwen/Qwen3.5-0.8B`를 씁니다.

- **GPU 메모리:** 가중치 전체를 학습하면 파라미터 하나에 약 16바이트(가중치와 기울기 4바이트씩, AdamW 상태 8바이트)가 들어 학습 상태만 약 12GB이고, 이 교안의 학습은 배치 4에서 최대 약 18GB를 쓰므로 이 단원은 24GB 이상 GPU에서 실행하고, 다 쓴 모델과 학습기는 `del`로 지웁니다.
- **파라미터 수:** `AutoModelForCausalLM`은 이미지 인코더를 빼고 글 모델만 불러오므로 이름의 0.8B보다 작습니다.
- **fast path 경고:** 일부 층을 빠르게 계산하는 전용 라이브러리가 없다는 뜻입니다. PyTorch로 같은 계산을 하므로 그대로 진행합니다.

In [ ]:
# 학습 전 모델의 Hugging Face 허브 이름
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저와 학습 전 모델(Base, Instruct 모델)을 불러옵니다. 첫 실행 때 약 1.7GB를 내려받습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,  # 파일은 bfloat16이지만, 학습할 때 작은 갱신이 반올림으로 사라지지 않게 float32로 불러옵니다
).to(device)

# SFT는 이 파라미터 전체를 조금씩 고칩니다. 이미지 인코더를 빼고 센 글 모델의 파라미터 수입니다
print(f"파라미터 수: {base_model.num_parameters():,}")

### 2-3. 답변 생성 함수

지난 시간 `chat()`과 같은 다섯 단계(messages, chat template, 토큰화, generate, 새 토큰만 디코드)를 함수 하나로 묶어, 학습 전후 모델에 같은 방법으로 묻습니다.

- **멈춤 토큰:** 이 모델은 기본으로 문서 끝 토큰 `<|endoftext|>`에서 멈추므로, 답 끝의 `<|im_end|>`에서 멈추도록 `eos_token_id`로 알려 줍니다.
- **그리디 생성:** 같은 질문에 늘 같은 답이 나와 학습 전후를 공정하게 비교할 수 있습니다. 작은 모델은 그리디에서 같은 말을 되풀이하기 쉬우므로, 서비스에 쓸 모델은 모델 카드가 권하는 샘플링 설정으로도 확인합니다(vLLM 단원).

In [ ]:
# 지난 시간 데이터의 시스템 메시지와 글자까지 같은 문장. 생성할 때도 같은 문장이어야 학습한 대로 답합니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_reply(model, question, system_prompt=SYSTEM_PROMPT, max_new_tokens=256):
    """질문 하나를 학습 데이터와 같은 chat template으로 감싸 모델이 새로 만든 답만 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
        enable_thinking=False,       # 빈 생각 블록(<think></think>)을 붙여 추론 과정 없이 바로 답합니다
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한. 답이 이보다 길면 끝까지 쓰기 전에 멈춥니다
            do_sample=False,                       # 매번 확률이 가장 높은 토큰을 골라(그리디 생성) 같은 질문에 같은 답이 나옵니다
            eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
            pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
        )

    # 입력 프롬프트 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

### 2-4. 지난 시간 데이터 불러오기

지난 시간에 원본 단위로 나눠 저장한 학습 파일과 검증 파일을 그대로 불러옵니다.

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# 지난 시간에 원본(origin_id) 단위로 나눠 저장한 학습 파일과 검증 파일을 그대로 읽습니다
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
val_rows = load_jsonl(data_dir / "본실습" / "labor_val.jsonl")
print("학습 행 수:", len(train_rows), "| 검증 행 수:", len(val_rows))

# 한 행: 원본 id(origin_id), 근거 조문(doc_id), 주제 태그(source), 질문, 정답 답변, messages
train_table = pd.DataFrame(train_rows)
display(train_table.head())

### 2-5. 학습 전 결과

학습 전후에 똑같이 물을 질문 여섯 개를 정하고, 학습 전 모델(Base)의 답을 모읍니다. 학습한 모델의 답은 뒤에서 같은 테이블에 열로 더해, 같은 질문의 답이 어떻게 바뀌는지 봅니다.

| 종류 | 개수 | 보는 것 |
|---|---|---|
| 학습 데이터 | 2 | 학습한 답을 따라 하는지 |
| 검증 데이터 | 1 | 학습에 쓰지 않은 노무 질문에도 맞게 답하는지 |
| 범위 밖 | 1 | 노동법으로 답할 수 없는 질문을 거절하는지(학습 데이터의 거절 답 행) |
| 일반 질문 | 2 | 원래 하던 상식 답을 지키는지 |

일반 질문은 시스템 프롬프트 없이 묻습니다. 노무 프롬프트로 물으면 거절해도 상식을 잃은 것인지, 배운 대로 거절한 것인지 가를 수 없기 때문입니다.

In [ ]:
# 지난 시간 데이터가 상담 범위 밖 질문에 단 거절 답(지난 시간 데이터 만들기와 같은 문장)
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 학습 데이터에서 정답 답변이 거절 답인 첫 행
for row in train_rows:
    if row["answer"] == REFUSAL_ANSWER:
        refusal_row = row
        break

# 노무와 상관없는 상식 질문(정답 답변은 reference_answer)
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 학습 전후에 똑같이 물을 질문 여섯 개와 그 종류
check_rows = [train_rows[300], train_rows[900], val_rows[39], refusal_row, general_rows[6], general_rows[24]]
check_kinds = ["학습 데이터", "학습 데이터", "검증 데이터", "범위 밖", "일반 질문", "일반 질문"]


def answer_check_questions(model):
    """여섯 질문에 model이 한 답을 같은 순서의 리스트로 돌려줍니다. 일반 질문은 시스템 프롬프트 없이 묻습니다."""
    answers = []
    for row, kind in zip(check_rows, check_kinds):
        if kind == "일반 질문":
            answers.append(generate_reply(model, row["question"], system_prompt=None))
        else:
            answers.append(generate_reply(model, row["question"]))
    return answers

In [ ]:
# 질문마다 정답 답변: 노무 행은 answer, 일반 질문은 reference_answer
reference_answers = []
for row in check_rows:
    if "answer" in row:
        reference_answers.append(row["answer"])
    else:
        reference_answers.append(row["reference_answer"])

# 종류, 질문, 정답 답변, Base 답을 한 테이블에 둡니다. 학습한 모델의 답은 뒤에서 이 테이블에 열로 더합니다
check_table = pd.DataFrame({
    "종류": check_kinds,
    "질문": [row["question"] for row in check_rows],
    "정답 답변": reference_answers,
    "Base 답변": answer_check_questions(base_model),
})
display(check_table)

**결과 읽기**
- **학습 데이터:** 강조 기호와 목록을 섞어 정답 답변보다 훨씬 길게 답하고 숫자도 틀립니다(배우자 유산 휴가 "1 년 또는 2 년", 조정안 기한 "20 일 이내". 정답은 5일, 60일).
- **범위 밖과 일반 질문:** 연말정산 질문을 거절하지 않고 날짜를 지어 답합니다. 한글을 만든 왕은 "실제로 존재하지 않는 인물", 숨 쉴 때 필요한 기체는 "호흡기"라고 답합니다.
- **검증 데이터:** [측정 뒤: 검증 행의 Base 답이 정답 답변(30분 이상의 휴게시간)과 비교해 길이, 숫자에서 어떤지]

SFT로 맞출 것은 상담 말투와 답 길이, 근거 조문의 숫자, 범위 밖 질문의 거절입니다.

## 3. 학습 전 점검

SFTTrainer는 만드는 순간 토큰화와 `labels`를 만들므로, `train()` 전에 학습 예시를 열어 볼 수 있습니다. GPU 시간을 쓰기 전에 데이터가 의도대로 바뀌었는지 먼저 확인합니다.

- **SFTTrainer:** trl 라이브러리의 학습기입니다. Hugging Face Trainer의 학습 루프에 chat template 적용과 답변 토큰만 손실로 계산하는 일을 더합니다.

### 3-1. prompt와 completion

messages를 그대로 주면 시스템 메시지와 질문까지 손실에 들어갑니다. 앞 메시지(system, user)를 `prompt`로, 배울 assistant 메시지를 `completion`으로 나눠 주면 답변 토큰만 손실로 계산합니다.

- **Dataset:** datasets 라이브러리의 테이블 모양 데이터입니다. SFTTrainer는 학습·검증 데이터를 이 형식으로 받습니다.

In [ ]:
from datasets import Dataset

def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}


# 학습 행과 검증 행을 모두 prompt, completion으로 바꿔 SFTTrainer가 받는 Dataset으로 만듭니다
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])

# 학습 예시 하나: prompt에는 system, user, completion에는 assistant 메시지가 들어 있습니다
display(train_dataset[0])

### 3-2. 토큰 길이와 max_length

`max_length`보다 긴 예시는 뒤쪽이 잘려 답 끝과 `<|im_end|>`를 잃고, 그 예시로는 답을 마치고 멈추는 법을 배우지 못합니다. 그래서 학습 전에 chat template으로 펼친 예시의 토큰 수를 셉니다.

- **이름 주의:** 예전 trl의 `max_seq_length`는 지금 trl에서 오류가 납니다.

In [ ]:
# 학습, 검증 예시의 전체 토큰 수(chat template으로 펼친 글)와 답변 토큰 수를 셉니다
full_lengths = []
answer_lengths = []
for row in train_rows + val_rows:
    text = tokenizer.apply_chat_template(row["messages"], tokenize=False)
    full_lengths.append(len(tokenizer(text)["input_ids"]))
    answer_lengths.append(len(tokenizer(row["answer"])["input_ids"]))

# 두 토큰 수의 분포(개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값)를 테이블로 봅니다
length_table = pd.DataFrame({"예시 토큰 수": full_lengths, "답변 토큰 수": answer_lengths})
display(length_table.describe().round(1))

In [ ]:
# 가장 긴 예시와 가장 긴 답변의 토큰 수
longest_example = length_table["예시 토큰 수"].max()
longest_answer = length_table["답변 토큰 수"].max()
print("가장 긴 예시:", longest_example, "토큰 | 가장 긴 답변:", longest_answer, "토큰")

# 학습 설정에 넣을 상한이 가장 긴 예시 이상이면 잘리는 예시가 없습니다
print("max_length 1024가 가장 긴 예시 이상인가:", 1024 >= longest_example)

# 상한을 128처럼 짧게 두면 끝이 잘리는 예시 수. 잘린 예시는 답변 끝(대화 끝 토큰)을 배우지 못합니다
n_cut_at_128 = (length_table["예시 토큰 수"] > 128).sum()
print(f"max_length 128로 자르면 끝이 잘리는 예시: {n_cut_at_128}개 / {len(length_table)}개")

**결과 읽기**
- **가장 긴 예시 210토큰, 가장 긴 답변 145토큰:** `max_length=1024`이면 잘리는 예시가 없습니다. 128로 자르면 1,484개 가운데 315개(약 21%)의 끝이 잘립니다.

### 3-3. 학습 설정과 학습기

학습 손실은 학습 데이터를 외워도 내려가므로, 학습에 쓰지 않은 검증 데이터의 손실을 함께 기록합니다. 검증 파일은 원본 단위로 나눠 같은 원본이 학습 파일에 없습니다.

- **검증 손실(eval_loss):** 검증 데이터로 계산한 같은 손실입니다. 학습 손실은 내려가는데 검증 손실이 오르면 학습 데이터를 외우는 중입니다(과적합).
- **모델을 고르는 기준은 아닙니다:** 검증 손실은 정답 답변 토큰에 준 확률이라, 모델이 직접 생성한 답이 맞는지는 알려 주지 않습니다. 쓸 모델은 교안 02에서 답을 채점해 고릅니다.


<img src="images/03_monitoring.png" width="1050" alt="제목은 '학습 손실과 검증 손실은 학습 기록입니다'입니다. 가로축 '갱신 횟수', 세로축 '손실' 그래프에서 '학습 손실'은 끝까지 내려가고 '검증 손실'은 내려가다 뒤에서 조금 다시 올라갑니다. 오른쪽 카드는 '둘 다 내려감'(학습이 정상으로 진행 중)과 '검증 손실만 오름'(학습 데이터를 외우는 중)입니다. 맨 아래 띠는 '모델은 손실이 아니라 교안 02에서 답을 채점해 고릅니다'입니다.">


`report_to=["mlflow"]`를 주면 학습기가 MLflow run을 열고 손실과 설정값을 기록합니다.

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("labor-sft")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

- **SFTConfig:** SFTTrainer의 설정 묶음입니다. Trainer 설정 이름(`learning_rate`, `num_train_epochs` 등)에 SFT용 설정(`max_length` 등)을 더했고, 인자의 뜻은 코드 주석에 있습니다. 기본으로 가중치는 float32로 두고 계산만 bfloat16으로 하는 혼합 정밀도(`bf16=True`)를 씁니다.

**학습률(lr)**
- **처음부터 학습할 때보다 훨씬 작게:** SFT는 이미 학습된 가중치를 조금만 고칩니다. 너무 크면 원래 능력을 덮어써 잃습니다(교안 02의 일반 능력 퇴행).
- **출발점:** 사전학습 최대 학습률의 1/10~1/20이 경험칙입니다(Llama 2 7B: 사전학습 3e-4, SFT 2e-5). 모르면 가중치 전체 SFT에서 널리 쓰는 1e-5~2e-5에서 시작하고, `SFTConfig` 기본값 2e-5가 이 범위입니다.

**첫 학습**
- **1에폭:** 데이터에 같은 사실을 표현만 바꾼 행이 있어 1에폭에도 같은 사실을 여러 번 봅니다.
- **저장하지 않음(`save_strategy="no"`):** 첫 학습과 4절 비교 학습은 답 비교에만 쓰고, 저장은 5절에서 에폭마다 합니다.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=str(output_dir / "labor_lr2e-5"),  # 학습기가 기록을 둘 폴더
    learning_rate=2e-5,                        # SFTConfig 기본값. 아래에서 더 작은 학습률(lr 5e-6)과 비교합니다
    num_train_epochs=1,                         # 학습 데이터 전체를 도는 횟수
    per_device_train_batch_size=4,              # 한 번 갱신에 쓰는 예시 수. 이 값에서 학습이 최대 약 18GB를 씁니다
    save_strategy="no",                        # 체크포인트를 저장하지 않습니다. 이 학습은 답 비교에만 씁니다
    seed=0,                                    # 데이터를 섞는 순서를 고정해 다시 실행해도 같은 순서로 학습합니다
    gradient_checkpointing=False,              # trl 기본은 켬(활성값을 버렸다가 역전파 때 다시 계산해 메모리 절약). 0.8B는 메모리가 충분해 끕니다
    max_length=1024,                          # 앞에서 가장 긴 예시 이상인지 확인한 상한
    eval_strategy="steps",                     # 정한 갱신 횟수마다 검증 손실을 계산합니다("epoch"이면 1에폭 학습에서 점이 너무 적습니다)
    eval_steps=50,                             # 검증 손실을 계산하는 간격(갱신 횟수)
    logging_steps=50,                          # 학습 손실도 같은 간격으로 기록해 두 곡선을 같은 step에 놓습니다
    eval_on_start=True,                        # 학습 전 검증 손실을 한 번 계산해 곡선의 출발점을 남깁니다
    report_to=["mlflow"],                      # 손실과 설정을 MLflow run에 남깁니다
    run_name="lr_2e-5",                        # MLflow에 보일 run 이름(학습률이 드러나게)
)

학습할 모델을 새로 불러와 학습기를 만듭니다. SFTTrainer는 받은 모델을 그 자리에서 고치므로 `base_model`과 따로 불러야 하고, 이 모델이 학습 뒤 lr 2e-5 모델이 됩니다. 학습은 아래 점검을 마친 뒤 시작합니다.

In [ ]:
# 학습할 모델을 학습 전 상태로 새로 불러옵니다. 학습이 끝나면 이 모델이 lr 2e-5 모델입니다
tuned_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# 학습기를 만들면 학습, 검증 데이터셋을 토큰화하고 labels를 만듭니다(아직 학습하지 않습니다)
trainer = SFTTrainer(
    model=tuned_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,     # 검증 손실을 계산할 데이터
    processing_class=tokenizer,   # chat template 적용과 토큰화에 쓸 토크나이저
)

### 3-4. 답변 토큰만 손실

학습기는 토큰마다 `input_ids`(그 자리의 토큰 id)와 `labels`(그 자리의 정답)를 만듭니다.

- **`labels`가 토큰 id인 자리:** 손실을 계산합니다(손실 식의 $y_t$). completion 쪽 토큰(답변, `<|im_end|>`, 줄바꿈)이 여기입니다.
- **`labels`가 -100인 자리:** 손실에서 빠집니다. PyTorch 교차 엔트로피가 -100을 건너뜁니다(`ignore_index=-100`). prompt 쪽 토큰이 여기입니다.


<img src="images/02_completion_loss.png" width="1050" alt="제목은 '답변 토큰에서만 손실을 계산합니다'입니다. 학습 예시 하나를 chat template으로 펼친 토큰 칸이 system, user, assistant 줄로 놓여 있습니다. system과 user 칸, assistant 머리와 빈 생각 블록 칸은 회색에 -100(손실 제외)이고, 답변 토큰과 <|im_end|>, 마지막 줄바꿈 칸은 빨강 테두리 초록에 '손실'입니다. 오른쪽 위 범례는 'labels -100: 손실 제외'와 '손실 계산'이고, 아래 띠에 손실 계산 토큰 수와 손실 제외 토큰 수가 적혀 있습니다.">

In [ ]:
# 학습기가 토큰화한 첫 학습 예시(학습 전이라 데이터만 만들어진 상태)
example = trainer.train_dataset[0]

# 토큰마다 글자, labels, 손실 대상인지를 테이블로 펼칩니다
token_table = pd.DataFrame({
    "토큰": [tokenizer.decode([token_id]) for token_id in example["input_ids"]],  # 한글 한 글자가 토큰 둘로 나뉘면 깨진 글자로 보입니다
    "labels": example["labels"],
})
token_table["손실 대상"] = token_table["labels"] != -100

# idxmax로 첫 손실 대상 토큰(답변 시작)의 자리를 찾아 7토큰 앞부터 17토큰을 봅니다(assistant 머리와 빈 생각 블록 포함)
answer_start = token_table["손실 대상"].idxmax()
display(token_table.iloc[answer_start - 7:answer_start + 10])

**결과 읽기**
- **답변 앞까지 `labels` -100:** 빈 생각 블록 `<think>` `</think>`도 prompt 쪽이라, 학습할 때와 생성할 때 답 앞부분이 같습니다.
- **답변 첫 토큰부터 `손실 대상` True:** 이 토큰들만 손실에 들어갑니다.

같은 예시의 마지막 토큰에서 대화 끝 토큰도 손실 대상인지 봅니다.

In [ ]:
# 같은 학습 예시의 마지막 4토큰
display(token_table.tail(4))

**결과 읽기**
- **`<|im_end|>`와 줄바꿈까지 `손실 대상` True:** 대화 끝 토큰도 학습하므로, 학습한 모델은 답을 마친 자리에서 멈춥니다.

## 4. 첫 학습과 학습률 비교

SFTConfig 기본 학습률 2e-5에 1에폭으로 첫 학습을 하고, 학습률만 5e-6로 줄여 한 번 더 학습해 비교합니다. 학습 하나만으로는 그 학습률이 알맞은지 판단할 기준이 없기 때문입니다.

### 4-1. 첫 학습(lr 2e-5)

학습 전 답은 2-5의 테이블에 모았으므로 `base_model`을 지워 GPU 메모리를 비우고 학습합니다.

In [ ]:
import gc

# 학습 전 답은 2-5의 테이블에 남았으므로, 학습 전 모델(base_model)이 차지하던 GPU 메모리를 비워 학습할 자리를 만듭니다
del base_model            # 모델을 가리키던 변수를 지웁니다
gc.collect()              # 더 이상 쓰지 않는 객체를 메모리에서 정리합니다
torch.cuda.empty_cache()  # PyTorch가 잡아 두었던 빈 GPU 메모리를 돌려줍니다

# 순전파 -> 손실 -> 역전파 -> 갱신을 반복하고, 정한 간격마다 검증 손실을 계산해 MLflow run에 기록합니다
# global_step은 실제 갱신 횟수, training_loss는 갱신 전체의 학습 손실 평균입니다
train_output = trainer.train()
display(train_output)

### 4-2. 학습 곡선

`trainer.state.log_history`에 쌓인 학습 손실과 검증 손실을 같은 갱신 횟수 축에 그립니다.

In [ ]:
# 학습기가 남긴 기록. 한 줄에 학습 손실(loss)이나 검증 손실(eval_loss) 가운데 하나가 있고, 마지막 줄은 학습 전체 요약입니다
# loss: 기록 간격(50번 갱신) 동안 갱신마다 계산한 배치 손실의 평균, eval_loss: 그 시점의 검증 데이터 전체 손실
log_table = pd.DataFrame(trainer.state.log_history)

# 학습 손실이 있는 줄과 검증 손실이 있는 줄을 나눕니다
train_log = log_table.dropna(subset=["loss"])
eval_log = log_table.dropna(subset=["eval_loss"])

# 학습 손실(파랑)과 검증 손실(빨강)을 갱신 횟수 축에 그립니다. 검증 손실은 학습 전(step 0)부터 있습니다
plt.figure(figsize=(7, 4))
plt.plot(train_log["step"], train_log["loss"], marker="o", color="#2563eb", label="학습 손실(loss)")
plt.plot(eval_log["step"], eval_log["eval_loss"], marker="o", color="#dc2626", label="검증 손실(eval_loss)")
plt.title("학습 곡선 (lr 2e-5)")
plt.xlabel("갱신 횟수 (step)")
plt.ylabel("손실 (교차 엔트로피)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**그래프 읽기**
- **검증 손실 약 2.8에서 약 1.4로:** 학습 내내 내려가고 오르는 구간이 없습니다. step 100 뒤로 학습 손실이 더 빨리 내려가 둘이 조금 벌어집니다.

### 4-3. lr 2e-5 모델의 답 모으기

2-5와 같은 여섯 질문을 lr 2e-5 모델에 묻고 같은 테이블에 열로 더해, Base 답 옆에서 학습 뒤 답을 봅니다.

In [ ]:
# 2-5와 같은 여섯 질문을 lr 2e-5 모델에 묻고, Base 답이 있는 테이블에 열로 더합니다
check_table["lr 2e-5 답변"] = answer_check_questions(tuned_model)
display(check_table)

**결과 읽기**
- **말투와 거절:** 노무 답이 "근로기준법 제○조에 따르면"처럼 한 문장 존댓말로 바뀌었고, 연말정산 질문은 학습한 거절 답과 같은 문장으로 거절합니다.
- **숫자는 틀림:** 학습한 질문인데도 배우자 유산 휴가를 "10일", 조정안 기한을 "3개월"로 답합니다. SFT는 말투를 잘 바꾸지만 사실을 정확히 떠올린다고 보장하지 못합니다.
- **일반 질문:** 숨 쉴 때 필요한 기체를 묻자 "1일 2회 각각 30분 이상의 유급휴가"를 되풀이해, 노무 데이터의 표현이 상식 답에 섞였습니다.
- **검증 데이터:** [측정 뒤: 검증 행의 lr 2e-5 답이 정답 답변과 결론, 숫자가 맞는지]

학습기 하나가 약 18GB라 24GB GPU에 둘을 함께 올릴 수 없으므로, 답을 모은 lr 2e-5 모델과 학습기는 저장하지 않고 지웁니다.

In [ ]:
# 학습기(AdamW 갱신 상태 포함)와 lr 2e-5 모델을 지워 다음 학습할 자리를 만듭니다
del trainer, tuned_model
gc.collect()
torch.cuda.empty_cache()

### 4-4. 학습률만 바꾼 학습(lr 5e-6)

학습률이 크면 가중치를 크게 바꿔 원래 하던 일반 답을 잃기 쉽습니다. 학습률만 5e-6로 줄인 학습을 같은 MLflow 실험에 남깁니다(run `lr_5e-6`, 폴더 `labor_lr5e-6`).

In [ ]:
# sft_config에서 학습률, 폴더, run 이름만 바꿉니다
small_lr_config = SFTConfig(
    output_dir=str(output_dir / "labor_lr5e-6"),  # 바꿈
    learning_rate=5e-6,                           # 바꿈: 첫 학습보다 작은 학습률
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_strategy="no",
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="lr_5e-6",                           # 바꿈
)

# 학습 전 상태의 모델을 새로 불러 같은 데이터로 학습기를 만듭니다
small_lr_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
small_lr_trainer = SFTTrainer(
    model=small_lr_model,
    args=small_lr_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

# 첫 학습과 같은 데이터, 같은 갱신 횟수로 학습합니다
small_lr_output = small_lr_trainer.train()
display(small_lr_output)

### 4-5. 두 학습의 검증 손실

두 학습의 검증 손실 곡선은 MLflow 화면에서 겹쳐 봅니다. VS Code에서 터미널을 하나 더 열고 단원 폴더에서 서버를 켭니다.

```bash
cd day70_SFT_지도파인튜닝
uv run mlflow server --backend-store-uri sqlite:///output/mlflow.db --port 5050
```

1. **주소 열기:** 브라우저에서 `http://127.0.0.1:5050`을 엽니다. RunPod에 VS Code로 접속해 있으면 **PORTS** 탭에서 5050 포트가 자동으로 이어집니다.
2. **run 목록:** 실험 `labor-sft`를 누르고 안내 말풍선을 **Got it**으로 닫은 뒤, 왼쪽 위 **Model training**을 누르면 `lr_2e-5`, `lr_5e-6` run이 보입니다.
3. **곡선 겹쳐 보기:** 목록 위 두 번째 버튼(차트 보기)을 누르고 **Search metric charts**에 `eval_loss`를 입력합니다.
4. **설정 보기:** run 이름을 누르면 설정값(Parameters)과 마지막 지표(Metrics)가 보입니다.

서버는 `Ctrl+C`로 끕니다. run이 보이지 않으면 단원 폴더에서 서버를 켰는지 확인합니다.

**그래프 읽기**
- **마지막 검증 손실 lr 2e-5 1.42, lr 5e-6 1.49:** step 200까지는 lr 5e-6이 더 낮다가 step 250부터 뒤집힙니다.
- 어느 학습률로 에폭을 늘릴지는 손실이 아니라 4-6의 답 비교로 정합니다.

### 4-6. 두 학습률의 답 비교

lr 5e-6 모델에도 같은 여섯 질문을 물어 테이블에 열로 더합니다.

In [ ]:
# 같은 여섯 질문을 lr 5e-6 모델에 묻고, Base, lr 2e-5 답이 있는 테이블에 열로 더합니다
check_table["lr 5e-6 답변"] = answer_check_questions(small_lr_model)
display(check_table)

**결과 읽기**
- **학습 데이터와 범위 밖:** lr 5e-6 모델도 한 문장 상담 말투로 답하고 연말정산 질문을 거절하지만, 숫자는 두 학습률 모두 틀립니다(배우자 유산 휴가 lr 5e-6 "10일 이내").
- **일반 질문:** lr 2e-5 모델은 "유급휴가"를 되풀이했지만, lr 5e-6 모델은 노무 표현 없이 "산소"로 맞힙니다.
- **검증 데이터:** [측정 뒤: 검증 행에서 두 학습률의 답이 정답 답변과 맞는지]

검증 손실은 lr 2e-5가 더 낮았지만(1.42, 1.49) 상식 답은 lr 5e-6이 지켰습니다. 그래서 5절은 lr 5e-6으로 에폭을 늘립니다. 질문 여섯 개라 경향만 보입니다.

In [ ]:
# 답을 모았으므로 lr 5e-6 모델과 학습기를 지워 다음 학습할 자리를 만듭니다(같은 학습률로 에폭을 늘린 학습은 5절에서 에폭마다 저장합니다)
del small_lr_model, small_lr_trainer
gc.collect()
torch.cuda.empty_cache()

## 5. 에폭 늘리기

4-6의 결과에 따라 lr 5e-6으로 4에폭을 학습하며 에폭마다 체크포인트를 저장하고, 에폭이 늘 때 답이 어떻게 바뀌는지 봅니다.

**에폭 정하기**
- **보통 1~3에폭:** 가중치 전체 SFT의 흔한 범위입니다(Llama 2 2에폭, [Stanford Alpaca](https://github.com/tatsu-lab/stanford_alpaca) 3에폭). 데이터가 적으면 더 돌리기도 합니다([LIMA](https://arxiv.org/abs/2305.11206) 1,000행에 15에폭).
- **늘릴 때 볼 것:** 학습 데이터의 답은 정답에 가까워지지만 일반 질문의 답을 잃을 수 있습니다. 그래서 에폭마다 저장해 둡니다.
- **이 교안의 4에폭:** 어디서 학습 데이터를 다 맞추는지 보려고 흔한 범위보다 한 에폭 더 학습합니다.

**저장과 학습 방법**
- **`save_strategy="epoch"`, `save_only_model=True`:** 에폭이 끝날 때마다 AdamW 상태를 빼고 가중치만 `checkpoint-<갱신 횟수>` 폴더에 저장합니다(하나에 약 3GB).
- **끝까지 학습:** 검증 손실은 에폭마다 기록만 하고, 그 값으로 학습을 멈추지 않습니다.
- **1에폭 체크포인트는 4절 lr 5e-6 모델과 다릅니다:** 학습률은 마지막 갱신까지 0으로 줄어드는데(`lr_scheduler_type` 기본값 `"linear"`), 4에폭 학습의 1에폭 끝에는 아직 크게 남아 있기 때문입니다.

### 5-1. 에폭을 늘려 학습

`sft_config`에서 학습률, 에폭, 검증과 저장 간격을 바꿉니다(`# 바꿈`).

In [ ]:
epoch_config = SFTConfig(
    output_dir=str(output_dir / "labor_epochs4_lr5e-6"),  # 바꿈: 에폭마다 checkpoint-<갱신 횟수> 폴더가 여기에 생깁니다(교안 02가 불러옵니다)
    learning_rate=5e-6,                           # 바꿈: 4절에서 비교한 작은 학습률
    num_train_epochs=4,                          # 바꿈: 중간에 멈추지 않고 끝까지 학습합니다
    per_device_train_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",                      # 바꿈: 에폭이 끝날 때마다 검증 손실을 계산해 기록합니다
    save_strategy="epoch",                      # 바꿈: 에폭이 끝날 때마다 체크포인트를 저장합니다
    save_only_model=True,                       # 바꿈: AdamW 갱신 상태 없이 가중치만 저장합니다
    logging_steps=50,                           # 학습 손실은 4절 첫 학습과 같은 간격으로 기록합니다
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="epochs4_lr_5e-6",                  # 바꿈
)

# 학습 전 상태의 모델을 새로 불러옵니다
epoch_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# 학습기는 만들 때 토크나이저의 끝 토큰(<|im_end|>)을 생성 설정의 멈춤 토큰에 더하므로, 저장한 체크포인트도 답 끝에서 멈춥니다
# 같은 데이터로 학습기를 만듭니다
epoch_trainer = SFTTrainer(
    model=epoch_model,
    args=epoch_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

epoch_output = epoch_trainer.train()
display(epoch_output)

### 5-2. 에폭별 학습 손실과 검증 손실

학습 전과 에폭마다 계산한 검증 손실을 학습 손실과 함께 에폭 축에 그립니다.

In [ ]:
# 학습기 기록에서 학습 손실이 있는 줄과 검증 손실이 있는 줄을 나눕니다
epoch_log_table = pd.DataFrame(epoch_trainer.state.log_history)
epoch_train_log = epoch_log_table.dropna(subset=["loss"])
epoch_eval_log = epoch_log_table.dropna(subset=["eval_loss"])

# 학습 손실(파랑)과 검증 손실(빨강)을 에폭 축에 그립니다
plt.figure(figsize=(7, 4))
plt.plot(epoch_train_log["epoch"], epoch_train_log["loss"], marker="o", color="#2563eb", label="학습 손실(loss)")
plt.plot(epoch_eval_log["epoch"], epoch_eval_log["eval_loss"], marker="o", color="#dc2626", label="검증 손실(eval_loss)")
plt.title("{EPOCH_CAP}에폭 학습 곡선 (lr {LR_COMPARE})")
plt.xlabel("에폭 (epoch)")
plt.ylabel("손실 (교차 엔트로피)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**그래프 읽기**
- **학습 손실:** 에폭 경계마다 계단처럼 떨어져 4에폭 끝에는 0에 가깝습니다.
- **검증 손실:** 1에폭이 가장 낮고 4에폭까지 조금씩 오릅니다. 학습 데이터를 외우는 모양이지만, 체크포인트는 버리지 않고 아래에서 답으로 봅니다.

### 5-3. 에폭별 체크포인트의 답

체크포인트는 `output/labor_epochs4_lr5e-6/checkpoint-<갱신 횟수>`에 저장되고, 교안 02가 이 이름으로 모두 불러옵니다. 저장된 폴더를 먼저 확인합니다.

In [ ]:
import math

# 에폭 하나의 갱신 횟수 = 올림(학습 행 수 / 배치). 마지막 배치는 남은 행만으로 한 번 갱신합니다
steps_per_epoch = math.ceil(len(train_dataset) / epoch_config.per_device_train_batch_size)

# 1에폭부터 4에폭까지 체크포인트 폴더 이름(폴더가 있는지도 확인합니다)
checkpoint_rows = []
for epoch in range(1, 5):
    checkpoint_dir = output_dir / "labor_epochs4_lr5e-6" / f"checkpoint-{steps_per_epoch * epoch}"
    checkpoint_rows.append({"에폭": epoch, "체크포인트": checkpoint_dir.name, "폴더 있음": checkpoint_dir.exists()})

checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)

# 에폭 학습의 모델과 학습기가 차지하던 GPU 메모리를 비워 체크포인트를 불러올 자리를 만듭니다
del epoch_model, epoch_trainer
gc.collect()
torch.cuda.empty_cache()

체크포인트를 하나씩 불러와 같은 여섯 질문에 답하게 하고, 다 쓰면 지운 뒤 다음 것을 불러옵니다.

In [ ]:
for checkpoint_row in checkpoint_rows:
    # 에폭 체크포인트를 학습 전 모델과 같은 방법(float32)으로 불러옵니다
    checkpoint_model = AutoModelForCausalLM.from_pretrained(output_dir / "labor_epochs4_lr5e-6" / checkpoint_row["체크포인트"], dtype=torch.float32).to(device)

    # 같은 여섯 질문의 답을 에폭 열로 더합니다
    check_table[f"{checkpoint_row['에폭']}에폭 답변"] = answer_check_questions(checkpoint_model)

    # 다 쓴 체크포인트를 지워 다음 체크포인트를 불러올 자리를 만듭니다
    del checkpoint_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# 종류, 질문, 정답 답변, Base 답 옆에 에폭별 답 열만 골라 봅니다(4절 두 학습률 모델의 답 열은 뺍니다)
epoch_columns = ["종류", "질문", "정답 답변", "Base 답변"]
for epoch in range(1, 5):
    epoch_columns.append(f"{epoch}에폭 답변")
display(check_table[epoch_columns])

**결과 읽기**
- **학습 데이터:** 배우자 유산 휴가는 1~3에폭에 "10일", "20일", "30일"로 틀리다가 4에폭에서 정답 답변과 글자까지 같아집니다. 조정안 기한은 3에폭부터 정답(60일)입니다.
- **범위 밖:** 1에폭은 연말정산 질문에 조문을 지어 답하고, 2에폭부터 거절합니다.
- **일반 질문:** 노무 표현은 어느 에폭에도 섞이지 않았고, 숨 쉴 때 필요한 기체는 2에폭부터 "산소", 한글을 만든 왕은 4에폭에서 "세종"으로 맞힙니다.
- **검증 데이터:** [측정 뒤: 검증 행의 에폭별 답이 정답 답변에 가까워지는지, 학습 데이터의 다른 답을 그대로 내놓는지]

검증 손실이 가장 낮은 1에폭과 학습한 답을 가장 잘 따라 하는 4에폭이 다릅니다. 어느 체크포인트를 쓸지는 교안 02에서 평가셋으로 채점해 정합니다.

## 6. 🖐️ 함께 따라하기: 전월세 상담 모델 학습하기

본실습의 학습 과정을 전월세 상담 데이터에 적용해 모델을 학습하고 저장합니다. 여기서 저장한 모델은 교안 02 따라하기가 씁니다.

**본실습과 다른 점**
- **데이터:** `data/따라하기/lease_train.jsonl`, `lease_val.jsonl`(지난 시간 따라하기의 전월세 데이터)
- **시스템 프롬프트:** `SYSTEM_PROMPT_LEASE`(학습 데이터의 시스템 메시지와 같은 문장)

**설정**
- 학습률 5e-6, 에폭 4(5절과 같음), `eval_steps`와 `logging_steps` 10(데이터가 적어 줄임)
- 배치 4, `max_length=1024`, `eval_strategy="steps"`, `eval_on_start=True`, `seed=0`, `gradient_checkpointing=False`(4절 첫 학습과 같음)
- run 이름 `lease_lr_5e-6`, 폴더 `output/lease_lr5e-6`

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 막히면 `정답/교안_01_정답.ipynb`의 같은 절을 실행해 `output/lease_lr5e-6` 폴더를 만듭니다(교안 02 따라하기가 씁니다).

### 6-1. 데이터 읽기

In [ ]:
# 전월세 데이터의 시스템 메시지와 글자까지 같은 문장
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."


# 지난 시간 따라하기에서 원본 단위로 나눠 저장한 전월세 학습, 검증 파일을 읽습니다
lease_train_rows = load_jsonl(data_dir / "따라하기" / "lease_train.jsonl")
lease_val_rows = load_jsonl(data_dir / "따라하기" / "lease_val.jsonl")
print("학습 행 수:", len(lease_train_rows), "| 검증 행 수:", len(lease_val_rows))

# 첫 행 몇 개를 테이블로 봅니다
lease_train_table = pd.DataFrame(lease_train_rows)
display(lease_train_table.head())

# 🖐️ 함께 따라하기: 행마다 to_prompt_completion으로 prompt, completion을 나눠 학습, 검증 Dataset을 만드세요(답변 토큰만 손실에 들어가게)
lease_train_dataset = ...
lease_val_dataset = ...
display(lease_train_dataset[0])

### 6-2. 설정과 학습

In [ ]:
# 🖐️ 함께 따라하기: 위 '설정'의 값으로 학습률, 에폭, max_length, 검증 손실 기록 인자를 채우세요
lease_config = SFTConfig(
    output_dir=str(output_dir / "lease_lr5e-6"),  # 학습기가 기록을 둘 폴더
    learning_rate=...,                 # 5절과 같은 학습률
    num_train_epochs=...,              # 5절과 같은 에폭 수
    per_device_train_batch_size=4,
    save_strategy="no",
    seed=0,
    gradient_checkpointing=False,
    max_length=...,                    # 본실습과 같은 상한
    eval_strategy=...,                 # 정한 갱신 횟수마다 검증 손실을 계산합니다
    eval_steps=...,                    # 갱신 횟수가 적어 기록 간격을 줄입니다
    logging_steps=...,                 # 학습 손실도 같은 간격으로 기록합니다
    eval_on_start=...,                 # 학습 전 검증 손실을 한 번 계산합니다
    report_to=["mlflow"],
    run_name="lease_lr_5e-6",
)

# 학습 전 상태의 모델을 새로 불러옵니다(본실습 모델과 따로 학습)
lease_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# 🖐️ 함께 따라하기: 학습기에 넘길 설정, 학습 데이터, 검증 데이터를 채우세요
lease_trainer = SFTTrainer(
    model=lease_model,
    args=...,
    train_dataset=...,
    eval_dataset=...,
    processing_class=tokenizer,
)

lease_output = lease_trainer.train()
display(lease_output)

### 6-3. 학습 곡선

In [ ]:
# 🖐️ 함께 따라하기: 학습기가 남긴 기록(lease_trainer.state.log_history)을 테이블로 만드세요
lease_log_table = ...

# 학습 손실이 있는 줄과 검증 손실이 있는 줄을 나눕니다
lease_train_log = lease_log_table.dropna(subset=["loss"])
lease_eval_log = lease_log_table.dropna(subset=["eval_loss"])

# 학습 손실(파랑)과 검증 손실(빨강)을 갱신 횟수 축에 그립니다
plt.figure(figsize=(7, 4))
plt.plot(lease_train_log["step"], lease_train_log["loss"], marker="o", color="#2563eb", label="학습 손실(loss)")
plt.plot(lease_eval_log["step"], lease_eval_log["eval_loss"], marker="o", color="#dc2626", label="검증 손실(eval_loss)")
plt.title("전월세 상담 학습 곡선")
plt.xlabel("갱신 횟수 (step)")
plt.ylabel("손실 (교차 엔트로피)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**그래프 읽기**
- **학습 손실 2.81에서 0.04로, 검증 손실 3.40에서 2.66으로:** 검증 손실은 step 60(2.48)에서 가장 낮고 그 뒤 오르내려 두 손실이 크게 벌어집니다. 답은 6-4와 교안 02 따라하기에서 봅니다.

### 6-4. 학습한 모델의 답

In [ ]:
# 학습한 질문 하나(학습 데이터 첫 행)와 학습에 없는 질문 하나(검증 데이터 첫 행)
lease_questions = [lease_train_rows[0]["question"], lease_val_rows[0]["question"]]
lease_references = [lease_train_rows[0]["answer"], lease_val_rows[0]["answer"]]

# 두 질문을 학습 데이터와 같은 시스템 프롬프트(SYSTEM_PROMPT_LEASE)로 lease_model에 묻습니다
lease_answers = []
for question in lease_questions:
    lease_answers.append(generate_reply(lease_model, question, SYSTEM_PROMPT_LEASE))

# 질문, 정답 답변, 학습 후 답변을 한 테이블에 나란히 둡니다
lease_answer_table = pd.DataFrame({"질문": lease_questions, "정답 답변": lease_references, "학습 후 답변": lease_answers})
display(lease_answer_table)

**결과 읽기**
- **학습한 질문:** 정답 답변과 글자까지 같은 문장으로 답합니다.
- **학습에 없는 질문(원룸 방 하나):** 학습 데이터 첫 행의 답을 그대로 내놓습니다. 결론("적용됩니다")은 정답과 같지만 근거로 다른 상황을 말합니다.

### 6-5. 저장

학습을 마친 모델은 `trainer.save_model(폴더)`로 저장합니다. 5절 체크포인트는 학습기가 에폭마다 저장했고, 여기서는 학습을 마친 뒤 한 번 저장합니다.

In [ ]:
# 🖐️ 함께 따라하기: lease_trainer의 모델을 output_dir / "lease_lr5e-6" 에 저장하세요(교안 02 따라하기가 불러옵니다)
...

# 저장된 파일 이름과 크기(MB)
lease_saved_files = []
for path in sorted((output_dir / "lease_lr5e-6").iterdir()):
    lease_saved_files.append({"파일": path.name, "크기(MB)": round(path.stat().st_size / 1024 ** 2, 1)})

lease_saved_file_table = pd.DataFrame(lease_saved_files)
display(lease_saved_file_table)

- **`model.safetensors`:** 학습한 가중치입니다. 가중치를 float32로 두고 학습(계산만 bf16)했으므로 float32로 저장되어 가장 큰 파일입니다.
- **`config.json`, `generation_config.json`:** 모델 구조와 생성 설정입니다.
- **`tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`:** 토크나이저와 chat template입니다. `from_pretrained(폴더)`로 모델과 함께 불러옵니다.
- **`training_args.bin`:** 학습에 쓴 설정입니다.

## 7. 핵심 정리

- **SFT:** 질문과 정답 답변 쌍으로 답변 토큰의 교차 엔트로피가 줄도록 가중치 전체를 더 학습합니다. 말투, 형식, 거절은 잘 배우지만 법 숫자 같은 사실은 틀리기 쉽습니다.
- **학습 전 점검:** prompt 토큰의 `labels`는 -100이고 답변과 `<|im_end|>`만 손실 대상이라, 답을 끝내는 자리까지 배웁니다. `max_length`는 가장 긴 예시(210토큰) 이상으로 둡니다.
- **GPU 메모리:** 가중치 전체 학습은 파라미터 하나에 약 16바이트(이 모델 약 12GB)가 들어, 다 쓴 모델과 학습기는 `del`로 지웁니다.
- **손실 기록:** `eval_strategy`, `eval_steps`, `report_to=["mlflow"]`로 검증 손실을 MLflow에 남깁니다. 손실 곡선은 학습이 정상인지 보는 기록이지 모델을 고르는 기준이 아닙니다.
- **같은 질문으로 비교:** 질문 여섯 개를 학습 전, 학습률 비교, 에폭별로 똑같이 물어 한 테이블에서 답이 바뀌는 모습을 봅니다. 검증 손실은 lr 2e-5가 더 낮았지만, 상식 답을 지킨 lr 5e-6으로 에폭을 늘렸습니다.
- **에폭 늘리기:** 4에폭 동안 에폭마다 체크포인트를 저장했고, 학습 데이터 질문의 답은 4에폭에서 정답 답변과 글자까지 같아졌습니다. 쓸 체크포인트는 교안 02에서 답을 채점해 고릅니다.

다음 시간에는 가중치 전체 대신 일부만 학습하는 LoRA를 배웁니다.